# 3 · Mechanism — behaviour, factor structure, reward-hacking  `[EVAL]`

**What the therapist actually does**, and whether the score gains are real MI skill or a style shift: MITI behaviour drift, the official MITI 4.2.1 competency thresholds, subscales, the rubric factor structure (one global-evaluation halo axis?), the reward-hacking synthesis + cross-checks (incl. the Q2 item-level reward composition), session shape, and ground-truth transcripts. Exports → `results/<VIEW>/figures|tables/3_mechanism/`. Outcomes → `1`, persona splits → `2`, stats tables → `6`.

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath("."))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)
import eda_analysis
from eda_analysis import stats, behavior, figures, plots

# ╔═══ VIEW — the one knob ════════════════════════════════════════════════════════╗
# "all" = every arm | "L0" = K=0 arms (PTO_LA0/GRPO_LA0) | "L5" = K=5 arms (thin, paused).
# Sets BOTH the arm filter AND the results root -> results/<VIEW>/figures|tables/<group>/.
# Edit the default for interactive use; render_views.py overrides it via the EDA_VIEW env var.
VIEW = os.environ.get("EDA_VIEW", "L0")

cfg = eda_analysis.EdaConfig(
    view=VIEW,                             # arm filter + results/<VIEW>/ root
    export_group="3_mechanism",            # topic family = this notebook's number
    selection="all",
    focus_arms=None, focus_metric="Q1Q2",
)
S = eda_analysis.notebook_setup(cfg)
FOCUS = cfg.focus_arms or sorted(S.SCORES.arm.unique())

In [ ]:
BEH = behavior.behavior_by_iter(S.ARMS)

## 1 · Behaviour drift  `[EVAL]`
**Purpose.** All 7 MITI behaviours **per therapist turn** (questions B3_Q, simple/complex reflections B4_SR/B5_CR, affirmations B6_AF, persuasion B2, giving-information B1_GI, seeking-collaboration B7_Seek) + text metrics
across iterations, per arm. Counts are length-normalized (÷ therapist turns) so a longer late-iteration
conversation doesn't inflate them. **Read:** B6_AF + turn length rising while B3_Q falls = the
affirmation/advice drift (warmth over skill). Combined grid `behavior_drift.png` + a per-metric zoom in `3_mechanism/behavior/<metric>.png`. (The MI-inconsistent side of the story — over-praise, unsolicited advice, etc. — is decomposed per-behaviour in §4d.)

In [ ]:
fig = plots.behavior_trajectory_grid(BEH)
if fig: eda_analysis.save_fig(fig, "behavior_drift", caption="All 7 MITI behavior rates PER THERAPIST TURN (B3_Q questions, B4_SR/B5_CR reflections, B6_AF affirmations, B2_Persuade persuasion, B1_GI giving-information, B7_Seek seeking-collaboration) + text metrics across iterations, all arms. Counts are length-normalized so longer late-iteration convs don't inflate them."); plt.show()
# NOTE: the semantic affirmation signal is the oracle-coded B6_AF (and MICI_OverPraiseRate) — the
# brittle lexical regex is only a directional sanity-check (see the over-praise cross-check below).
# Behavior counts are shown per therapist turn (B*_per_turn) so the drift isn't a length artifact.
BM = [m for m in ["B3_Q_per_turn", "B4_SR_per_turn", "B5_CR_per_turn", "B6_AF_per_turn",
                  "B2_Persuade_per_turn", "B1_GI_per_turn", "B7_Seek_per_turn",
                  "RtoQ", "Empathy", "loop", "q_per_turn"] if m in BEH.columns]
BT = BEH[["arm", "iteration"] + BM].round(3).sort_values(["arm", "iteration"]).reset_index(drop=True)
display(BT)
eda_analysis.save_table(BT, "behavior_by_iter", caption="Mean behavior metrics per (arm, iteration): all 7 MITI behavior counts per therapist turn + text metrics — one merged table, all arms.")

# Per-metric zoom — the 3_mechanism/behavior/ subfolder companion to the combined grid above.
for m in BM:
    figm = plots.single_behavior_trajectory(BEH, m)
    if figm is None:
        continue
    eda_analysis.save_fig(figm, m, group="3_mechanism/behavior",
                          caption=f"{eda_analysis.display_label(m)} across iterations, all arms — per-metric zoom of behavior_drift.")
    plt.show()

## 2 · Subscale trajectories  `[EVAL]`
One line per WAI/MITI sub-scale, a panel per (parent, arm) — do some components (Bond/Empathy) climb faster than others (Goal/ChangeTalk)? Combined grid `subscale_trajectories.png` + a per-parent zoom in `3_mechanism/subscales/<parent>.png`.

In [ ]:
SUB = eda_analysis.load_subscales(S.ARMS)
fig = plots.subscale_trajectory_grid(SUB, min_iters=3)
eda_analysis.save_fig(fig, "subscale_trajectories", caption="WAI-SR + MITI global subscale means across iterations; one panel per (parent, arm), arms with <3 scored iters omitted."); plt.show()

# Per-parent zoom — the 3_mechanism/subscales/ subfolder companion to the combined grid above.
for parent in ["WAI-SR", "MITI"]:
    figp = plots.subscale_trajectory_grid(SUB, parents=(parent,), min_iters=3)
    if figp is None:
        continue
    eda_analysis.save_fig(figp, parent, group="3_mechanism/subscales",
                          caption=f"{parent} subscale trajectories across iterations; one panel per arm.")
    plt.show()

## 2b · Official MITI 4.2.1 competency thresholds — absolute anchor  `[EVAL]`
**Purpose.** Everything else in the EDA is *relative* (vs base, vs the other arm). This section anchors the therapist against the **official MITI 4.2.1 competency standards** (Moyers, Manuel & Ernst 2014, manual rev. 2015, §I): the manual's four summary scores — `R:Q` (fair ≥ 1:1, good ≥ 2:1), `%CR` (≥ 40% / 50%), **Technical global** = (Cultivating Change Talk + Softening Sustain Talk)/2 (≥ 3 / 4) and **Relational global** = (Partnership + Empathy)/2 (≥ 3.5 / 4) — computed for free from the already-scored MITI globals + counts (note: Technical/Relational are the manual's 2-global splits, *not* our 4-global `MITI_GlobalMean`).

**Read:** training moves both arms from *below basic competence* to **fair-to-good on the global ratings** (Relational crosses "good"), but **neither arm reaches "good" on the technique ratios** — and GRPO's late `R:Q` jump is the *pathological* route (the iter-10 question collapse shrinks the denominator, not more reflection; cross-check §1's `B3_Q/turn` and §4b). **Caveats (the manual's own + ours):** thresholds are expert opinion without normative validation, and are defined for ~20-min human audio sessions — short text chats are out-of-domain. Use as an anchor, not a certification.

In [ ]:
# 2b · The manual's 4 summary scores vs its fair/good thresholds (constants.MITI_THRESHOLDS).
PROF = behavior.miti_proficiency_by_iter(S.ARMS)
fig = plots.miti_threshold_panel(PROF, palette=S.PALETTE)
if fig is not None:
    eda_analysis.save_fig(fig, "miti_proficiency_thresholds", caption="The 4 official MITI 4.2.1 summary scores (R:Q, %CR, Technical global, Relational global) across iterations vs the manual's fair/good competency thresholds (dashed). Both arms cross 'good' on Relational but neither reaches 'good' on the technique ratios; GRPO's late R:Q jump is driven by the iter-10 question collapse (smaller denominator), not more reflection. Thresholds are expert opinion (MITI 4.2.1 manual) and defined for ~20-min human sessions."); plt.show()
    PROF_T = plots.miti_threshold_table(PROF)
    display(PROF_T)
    eda_analysis.save_table(PROF_T, "miti_threshold_verdicts", caption="Base vs final MITI 4.2.1 summary scores per arm with the manual's competency verdict per score (✓good / ✓fair / ✗ below basic competence).")
else:
    print("MITI not scored yet — run Run_Eval.ipynb to populate the proficiency panel.")

## 3 · Rubric factor structure — two metric families  `[EVAL]`
**Purpose.** Are the rubrics independent skills or one halo? The battery was *designed* as two families, but the data **revises the boundary** — made explicit below in the correlation heatmap (heavy block divider) and the loadings bars (blue vs orange):

- **Global-evaluation (halo) cluster — one factor (blue).** `Q1+Q2`, `WAI-SR`, `CSQ-8`, `MI-SAT`, `MITI` (globals) — subjective satisfaction / alliance / integrity ratings. An **empirical redundancy set, not one official construct** (Q2 and WAI-SR even overlap by design — both alliance measures; per-instrument provenance: `METRICS_REFERENCE.md` §1). On their own they collapse to **one factor (PC1≈91%)** — the single-oracle halo — so "all global-eval rubrics rise together" reflects a single latent axis, **not** multi-skill gain.
- **Intended orthogonal axes (orange)** — built to load *off* that factor: **`PCT`** (patient change-talk proportion — the MI *goal*), **`MICI ↓`** (MI-inconsistent / sycophancy rate, lower = better), and the *free* objective MITI-proficiency ratios **`R:Q` / `%CR` / `%MICO`** (derived from existing behavior counts, no rescoring).

**Finding — PCT is NOT the clean orthogonal axis it was intended as.** Empirically `PCT` (change-talk proportion) loads **with the halo family** (Spearman ρ≈0.79–0.94; high PC1 loading): warmer, more satisfying sessions also elicit more patient change-talk, so PCT does not isolate MI *technique*. The **genuine second axis** is **`MICI ↓` + the MITI ratios `R:Q` / `%CR` / `%MICO`** (objective reflection-vs-persuasion technique). **Read:** adding the orthogonal axes still drops PC1 from ≈91% to **≈55%** — a real second dimension exists — but it is defined by MI-inconsistency + technique ratios, **not** by PCT. (Reward faithfulness lives in `4_Training_and_Reliability`.)

In [ ]:
# Expand the scores with the FREE objective MITI-proficiency ratios (+ PCT/MICI if already scored).
SCO = eda_analysis.add_derived_mitiprof_rows(S.SCORES, S.ARMS)
EXPANDED = [m for m in (eda_analysis.WARMTH_RUBRICS + eda_analysis.ORTHOGONAL_METRICS)
            if m in SCO.questionnaire.unique()]
print("metrics in correlation/PCA:", EXPANDED)

# Inter-rubric correlation over the EXPANDED set (global-eval halo rubrics + orthogonal axes).
fig = plots.rubric_correlation_heatmap(SCO, metrics=EXPANDED)
eda_analysis.save_fig(fig, "rubric_correlation", caption="Spearman correlation among rubric + orthogonal-axis scores (per conversation, pooled). The global-eval (halo) rubrics block together; PCT loads WITH them (rho~0.79-0.94, not orthogonal as intended), while MICI + the MITI ratios R:Q/%CR/%MICO form the genuine second family."); plt.show()

# PCA: pooled + per arm. Does adding orthogonal axes drop the dominant PC1 share?
pca_all = stats.rubric_pca(SCO, metrics=EXPANDED)
if pca_all["explained_variance_ratio"]:
    print(f"\nPOOLED: PC1 = {pca_all['explained_variance_ratio'][0]:.1%}  "
          f"(over {len(pca_all['metrics'])} metrics: {pca_all['metrics']})")
    print(f"        PC1 loadings = {pca_all['pc1_loadings']}")
    print("        -> global-eval (halo) rubrics (+ PCT, which loads WITH them) load high on PC1;")
    print("           only MICI + the MITI ratios R:Q/%CR/%MICO stay near-zero = the genuine second axis.")
for arm in sorted(SCO.arm.unique()):
    p = stats.rubric_pca(SCO[SCO.arm == arm], metrics=EXPANDED)
    if p["explained_variance_ratio"]:
        print(f"{arm}: PC1 = {p['explained_variance_ratio'][0]:.1%}  loadings={p['pc1_loadings']}")

### What PC1 measures — factor loadings  `[EVAL]`
Each metric's loading on the principal components. The 5 global-eval rubrics all load high on **PC1** (~0.44 each) → they are essentially **one** halo factor ('good warm therapist' as a single judgment) — and **`PCT` loads high on PC1 with them** (ρ≈0.79–0.94), so despite the design intent it is *not* orthogonal. Only the objective technique ratios **`R:Q` / `%CR` / `%MICO`** and **`MICI ↓`** load ~0 on PC1 and define **PC2** — they are the axis that measures something the halo doesn't.

In [ ]:
fig = plots.factor_loadings_bars(SCO, metrics=EXPANDED, components=("PC1","PC2"))
if fig is not None:
    eda_analysis.save_fig(fig, "factor_loadings", caption="Each metric's PCA loading: the 5 global-eval (halo) rubrics load high on PC1 (one shared factor) and PCT loads high WITH them (~0.79-0.94, not orthogonal as intended); only the MITI ratios R:Q/%CR/%MICO and MICI load ~0 on PC1 and define PC2."); plt.show()
else:
    print("factor loadings need >=2 metrics with enough rows.")

## 4 · Reward-hacking — the headline concern  `[EVAL]`
The synthesis of §1–§3: the warmth **reward proxy** climbs, but the signals *outside* the reward expose the cost.

**Two confounds this section is built to survive.** (i) **Reward = outcome:** Q1+Q2 is *both* the training reward *and* a headline eval metric — so "Q1+Q2 went up" is partly circular and can't, by itself, prove skill gain. (ii) **Shared oracle:** the simulated patient *and* the grader are the **same** model (gpt-4o-mini-2024-07-18), which couples the generator and the evaluator and can inflate the patient-perspective ratings. The reward-hack conclusion therefore does **not** rest on Q1+Q2. It rests on signals that break these confounds, strongest first:
- **Deterministic text metrics** (turn length ↑, loop %, question rate ↓; `behavior.py` regex) — computed from the transcript with **no oracle at all**, so they break **both** confounds. These are the load-bearing evidence.
- **Oracle-scored but *un-rewarded* axes** (`MICI ↑`, `PCT` ~flat, the MITI ratios `R:Q`/`%CR`/`%MICO`) — not part of the reward, so they break the reward=outcome circularity; they still share the oracle *model*, so they corroborate rather than prove.

- **4a** — one twin-axis frame: warmth ↑ **and** MI-inconsistency ↑ *together* while patient change-talk stays ~flat → "all rubrics up" is **not** multi-skill.
- **4b** — question-rate cross-check: regex literal-`?` rate vs the oracle question-*function* count (both /turn, same denominator). They agree on **direction** (questions fall) but **diverge in magnitude** — the regex collapses ~7× for GRPO while the oracle drops ~1.6×, because late affirmation/advice turns carry question-function without a literal `?`. Syntax vs function, **not a unit bug** (merge is conv-aligned 96/96).
- **4c** — over-praise cross-check: the lexical marker rate agrees directionally with the oracle's `MICI_OverPraiseRate`.

(The annotated per-metric curves live in `1_Outcomes/trajectories/`; the persona split of the regression in `2_Heterogeneity`.)

In [ ]:
# 4a · The reward-hack in ONE frame — Q1+Q2 reward proxy (left, 1-5) vs MICI/PCT (right, 0-1), twin-axis per arm.
fig = plots.reward_hack_panel(S.SCORES, arms=FOCUS, palette=S.PALETTE)
if fig is not None:
    eda_analysis.save_fig(fig, "reward_hack_panel", caption="Twin-axis per arm: the global-eval reward proxy Q1+Q2 (left, 1-5) rises while MI-Inconsistency (right, higher=worse) rises with it and Patient Change-Talk (right, the actual MI goal) stays ~flat — 'all rubrics up' is not multi-skill."); plt.show()
else:
    print("no focus arms present.")

In [ ]:
# 4b · Question-rate cross-check — deterministic ?-count/turn vs oracle MITI B3_Q/turn (same denominator).
# NOT a unit bug: both are questions-per-therapist-turn, but the numerators are different constructs —
# literal '?' SYNTAX vs oracle question-FUNCTION. They agree on direction (questions fall) but diverge in
# magnitude for GRPO (regex collapses ~7x, oracle ~1.6x): late affirmation/advice turns carry
# question-function the oracle credits without a literal '?'. The widening gap IS the drift signature.
QRC = behavior.question_rate_crosscheck(S.ARMS)
fig = plots.question_rate_crosscheck(QRC)
if fig is not None:
    eda_analysis.save_fig(fig, "question_rate_crosscheck", caption="Questions per therapist turn two ways: regex literal '?' vs oracle MITI question-function (B3_Q/turn, same denominator). Both fall, but for GRPO the regex rate collapses ~7x while the oracle count drops only ~1.6x — the widening gap = the affirmation/advice drift (declarative prompts carry question-function but no literal '?'). Syntax vs function, not a unit error."); plt.show()
else:
    print("MITI not scored yet — question-rate cross-check empty.")

In [ ]:
# 4c · Over-praise cross-check: deterministic lexical marker rate vs the oracle's MICI_OverPraiseRate.
# Validates the DIRECTION of the demoted regex against the professional coder (same role loop% plays
# for degeneration). Empty until the MICI questionnaire is scored via Run_Eval.
XC = behavior.overpraise_crosscheck(S.ARMS)
if XC.empty:
    print("MICI not scored yet — run Run_Eval.ipynb with QUESTIONNAIRE_FILTER=['PCT','MICI'] to populate this cross-check.")
else:
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.scatterplot(XC, x="lex_overpraise_marker_rate", y="MICI_OverPraiseRate",
                    hue="arm", palette=figures.arm_palette(sorted(XC.arm.unique())), s=60, ax=ax)
    rho = XC[["lex_overpraise_marker_rate", "MICI_OverPraiseRate"]].corr(method="spearman").iloc[0, 1]
    ax.set_title(f"Over-praise: lexical marker vs oracle (Spearman rho={rho:.2f})")
    ax.set_xlabel("lexical over-praise marker rate (regex)"); ax.set_ylabel("oracle MICI over-praise rate")
    figures.relabel_legend(ax)
    fig.tight_layout()
    eda_analysis.save_fig(fig, "overpraise_crosscheck", caption="Per-(arm,iteration) deterministic lexical over-praise marker rate vs the oracle-coded MICI over-praise rate — a directional sanity-check on the demoted regex."); plt.show()

### 4d · MI-inconsistent behaviour detail (MICI) — *which* harmful move drives the rise?  `[EVAL]`
The reward-hack panel (§4a) reduced MI-inconsistency to one aggregate rate. Here it is broken into the **severity global (1-5)** + each of the **6 MI-inconsistent behaviours per therapist turn** (higher = worse), so the rise can be attributed. **Read:** the increase is **almost entirely over-praise (sycophancy)** — `MICI_OverPraise/turn` climbs ~20–35× while confront / warn / judge stay ~0 and advise-without-permission sits at a flat background — confirming the affirmation-drift reading of §1 from the *negative* side, and it runs away hardest for GRPO at iter 10. Combined grid `mici_behavior_detail.png` + per-behaviour zoom in `3_mechanism/mici/`. Empty until MICI is scored.

In [ ]:
# 4d · MICI severity + each of the 6 MI-inconsistent behaviours PER THERAPIST TURN (higher = worse).
MICI_BEH = behavior.mici_behavior_by_iter(S.ARMS)
if MICI_BEH.empty:
    print("MICI not scored yet — run Run_Eval.ipynb with QUESTIONNAIRE_FILTER=['MICI'] to populate this.")
else:
    MICI_RATES = [c for c in MICI_BEH.columns if c.endswith("_rate")]  # the 6 per-behaviour rates
    fig = plots.behavior_trajectory_grid(
        MICI_BEH, palette=S.PALETTE, metrics=["MICI_Severity", "MICI_Rate"] + MICI_RATES,
        title="MI-inconsistent behaviour detail (MICI) — severity + per-therapist-turn rates (higher = worse)")
    if fig: eda_analysis.save_fig(fig, "mici_behavior_detail", caption="MICI severity global (1-5) + total and per-behaviour MI-inconsistent rates PER THERAPIST TURN across iterations, all arms. Higher = worse. Isolates WHICH harmful move drives the MI-inconsistency rise: over-praise dominates (~20-35x), while confront/warn/judge stay ~0 and advise-without-permission sits at a flat background."); plt.show()
    for m in ["MICI_Severity"] + MICI_RATES:
        figm = plots.single_behavior_trajectory(MICI_BEH, m, palette=S.PALETTE)
        if figm is not None:
            eda_analysis.save_fig(figm, m, group="3_mechanism/mici",
                                  caption=f"{eda_analysis.display_label(m)} across iterations, all arms — per-behaviour zoom of mici_behavior_detail.")
            plt.close(figm)
    MT = MICI_BEH.drop(columns=["method", "K"]).round(3).sort_values(["arm", "iteration"]).reset_index(drop=True)
    display(MT)
    eda_analysis.save_table(MT, "mici_behavior_by_iter", caption="Mean MICI severity + total and per-behaviour MI-inconsistent rates per therapist turn, per (arm, iteration), all arms.")

### 4e · Patient change-talk detail (PCT) — did the *patient* actually move?  `[EVAL]`
The reward-hack panel used one aggregate change-talk proportion. Here are the **3 patient-perspective globals** (Importance / Confidence / Readiness, 1-5) + the **utterance-type proportions** (% change / % sustain / % neutral of patient turns) — the actual MI *target*, scored from the patient side. **Read:** patient motivation moves **modestly and mostly for PTO** (Importance/Readiness up, sustain-talk down); GRPO's late regression shows here too (globals dip at iter 9-10). Combined grid `pct_patient_detail.png` + per-signal zoom in `3_mechanism/pct/`. Empty until PCT is scored.

In [ ]:
# 4e · PCT patient globals (1-5) + change/sustain/neutral proportions of patient utterances.
PCT_BEH = behavior.pct_behavior_by_iter(S.ARMS)
if PCT_BEH.empty:
    print("PCT not scored yet — run Run_Eval.ipynb with QUESTIONNAIRE_FILTER=['PCT'] to populate this.")
else:
    PCT_GLOB = [c for c in ["PCT_Importance", "PCT_Confidence", "PCT_Readiness"] if c in PCT_BEH.columns]
    PCT_PROP = [c for c in PCT_BEH.columns if c.endswith("_prop")]
    fig = plots.behavior_trajectory_grid(
        PCT_BEH, palette=S.PALETTE, metrics=PCT_GLOB + ["PCT_ChangeProp"] + PCT_PROP,
        title="Patient change-talk detail (PCT) — globals (1-5) + patient-utterance proportions")
    if fig: eda_analysis.save_fig(fig, "pct_patient_detail", caption="PCT patient-perspective globals (Importance/Confidence/Readiness, 1-5) + change/sustain/neutral proportions of patient utterances across iterations, all arms. The actual MI target scored from the patient side: motivation moves modestly (more for PTO), while GRPO's late-iteration regression appears as a global dip."); plt.show()
    for m in PCT_GLOB + ["PCT_ChangeProp"] + PCT_PROP:
        figm = plots.single_behavior_trajectory(PCT_BEH, m, palette=S.PALETTE)
        if figm is not None:
            eda_analysis.save_fig(figm, m, group="3_mechanism/pct",
                                  caption=f"{eda_analysis.display_label(m)} across iterations, all arms — per-signal zoom of pct_patient_detail.")
            plt.close(figm)
    PT = PCT_BEH.drop(columns=["method", "K"]).round(3).sort_values(["arm", "iteration"]).reset_index(drop=True)
    display(PT)
    eda_analysis.save_table(PT, "pct_patient_by_iter", caption="Mean PCT globals + patient-utterance proportions per (arm, iteration), all arms.")

### 4f · Q2 reward composition — *which* alliance items does the optimizer exploit?  `[EVAL]`
**Purpose.** Q1+Q2 is the training reward, and Q2's 17 items are not equal: items 1/2/3/10 reward therapist **self-disclosure** ("revealed what he was thinking", "shared his feelings", "let me know when he was happy or sad") — behavior MI does not prescribe — beside empathy, fluency, warmth/closeness and non-judgment items. Since per-item scores are already on disk (`Q2_1..Q2_17`, no oracle re-run), we can decompose **which reward components the optimizer actually moved**: the endpoint Δ-vs-base per item + when each face-content group takes off. Item groups are OUR analytical reading of the items (`constants.Q2_ITEM_GROUPS`), **not** a validated subscale structure.

**Read:** if self-disclosure / warmth-closeness items climb hardest while e.g. "did not judge me" moves less, the reward *composition itself* incentivizes the emotive/over-praise drift of §1/§4d — tracing the reward-hack to specific reward components rather than to the optimizer alone.

In [ ]:
# 4f · Q2 per-item reward composition (per-item columns already stored — no oracle re-run).
Q2I = eda_analysis.load_q2_items(S.ARMS)
if Q2I.empty:
    print("Q2 not scored yet — run Run_Eval.ipynb to populate the item-level view.")
else:
    Q2D = stats.q2_item_endpoint_deltas(Q2I)
    fig = plots.q2_item_delta_bars(Q2D)
    if fig: eda_analysis.save_fig(fig, "q2_item_drivers", caption="Endpoint Delta vs base per Q2 item (one panel per arm, shared Delta scale), colored by face-content item group (analytical grouping, not a validated subscale). Which alliance items the Q1+Q2 training reward actually moved — self-disclosure items ('revealed his thinking') top the list, i.e. the reward composition itself incentivizes the emotive drift."); plt.show()
    fig = plots.q2_item_group_trajectory(Q2I)
    if fig: eda_analysis.save_fig(fig, "q2_item_group_trajectories", caption="Mean Q2 item-group score across iterations per arm (+-95% CI; face-content groups, analytical not validated) — when each exploited reward component takes off."); plt.show()
    Q2T = Q2D.sort_values(["arm", "delta"], ascending=[True, False]).reset_index(drop=True)
    display(Q2T)
    eda_analysis.save_table(Q2T, "q2_item_deltas", caption="Per (arm, Q2 item): base mean, final mean and Delta, with the item's short text + face-content group — the reward-composition table behind q2_item_drivers.")

## 5 · Session end + length  `[EVAL]`
**Purpose.** Mean conversation length across iterations + how sessions terminate. **Read:** rising
length tracks the advice drift; the end-reason mix is a degeneration health-check. (Inline only — not exported.)

In [ ]:
TEXT = behavior.text_metrics(S.ARMS, attach_persona=False)
fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(TEXT, x="iteration", y="conv_len", hue="arm", palette=figures.arm_palette(sorted(TEXT.arm.unique())), marker="o", ax=ax)
ax.set_title("Mean conversation length (utterances)"); plt.show()
rows = []
for a in S.ARMS:
    for k in a.iters:
        cdir = a.conv_dir(k)
        for fn in (os.listdir(cdir) if cdir and os.path.isdir(cdir) else []):
            if fn.startswith("conversation_") and fn.endswith(".csv"):
                try: dd = pd.read_csv(os.path.join(cdir, fn))
                except Exception: continue
                rows.append({"arm": a.label, "ended_by": str(dd["session_ended_by"].iloc[0]) if "session_ended_by" in dd else "NA"})
if rows:
    display(pd.DataFrame(rows).groupby(["arm", "ended_by"]).size().rename("n").reset_index()
            .pivot_table(index="arm", columns="ended_by", values="n", fill_value=0))

## 6 · Persona-matched transcripts  `[EVAL]`
**Purpose.** The same patient persona's conversation early / mid / late per arm — the qualitative drift
in actual words (true-persona recovery makes the match exact across the per-iteration shuffle). Print-only.

In [ ]:
from eda_analysis import persona_order
def file_of_persona(seed, k, pid, n=96): return persona_order(seed, k, n).index(pid)
PERSONA = 0
print("persona", PERSONA, "=", eda_analysis.canonical_personas().loc[PERSONA].to_dict(), "\n")
for arm in S.ARMS:
    iters = arm.iters
    pick = sorted(set([iters[0], iters[len(iters) // 2], iters[-1]]))
    print(f"\n############  {arm.label}  ############")
    for k in pick:
        cdir = arm.conv_dir(k)
        if not cdir: continue
        fi = file_of_persona(arm.seed, k, PERSONA)
        fp = os.path.join(cdir, f"conversation_{fi}.csv")
        if not os.path.exists(fp): continue
        d = pd.read_csv(fp); th = d[d.role == "therapist"]["conversation"].astype(str).tolist()
        print(f"==== {arm.label} model_iter_{k} (conv_{fi}) — {len(th)} therapist turns ====")
        for t in th[1:4]:
            print("   •", " ".join(t.split())[:240]); print()

## 7 · How to read this notebook
- The **affirmation-drift** signature (B6_AF + turn length up, B3_Q down, §1) is the warmth-over-skill flag — compare PTO vs GRPO onset at matched late iters (§1 shows all 7 MITI behaviour counts per therapist turn, so longer convs don't inflate them).
- **§2b anchors absolutely:** vs the official MITI 4.2.1 competency thresholds, training reaches fair-to-good on the *global* ratings but **neither arm reaches "good" on the technique ratios** (R:Q / %CR) — and GRPO's late R:Q "improvement" is the pathological fewer-questions route.
- **Factor structure** (§3): the 5 global-eval rubrics alone share a dominant **PC1** (≈91%) — the single-oracle halo; adding the orthogonal axes drops PC1 to ≈55%, but **PCT loads WITH the halo** (ρ≈0.79–0.94, not orthogonal as intended) — the genuine second factor is **`R:Q` / `%CR` / `%MICO` + `MICI ↓`** (objective technique + MI-inconsistency).
- **§4 Reward-hacking** is the synthesis, built to survive the reward=outcome and shared-oracle confounds: the load-bearing evidence is the **deterministic** text metrics (no oracle), corroborated by the un-rewarded oracle axes (MICI ↑, PCT flat). **§4d** decomposes MI-inconsistency per behaviour (the rise is ~all over-praise), **§4e** decomposes patient change-talk (globals + change/sustain/neutral proportions), and **§4f** decomposes the *reward itself* (which Q2 items the optimizer moved — self-disclosure items top the list) — so neither the aggregates nor the reward are black boxes.
- **Transcripts** (§6) are the ground truth.
- _(Outcome curves → `1_Outcomes`; persona splits → `2_Heterogeneity`; reward faithfulness → `4_Training_and_Reliability`; exact stats → `6_Stats`.)_

In [ ]:
print("index ->", eda_analysis.build_index())